# 7. Export final

Última etapa do pipeline: valida o output já gerado por
`scripts/build_items_json.py` / `scripts/build_buildings_json.py` contra o
contrato de `src/lib/types.ts`, roda a validação de ciclo e de referência
sobre o **resultado final** (não mais o dado bruto), e mostra um diff
semântico contra o HEAD do git - antes de considerar o output pronto para
o app consumir.

Este notebook não gera `src/data/*.json` - isso é feito pelos scripts /
notebooks em `notebooks/build_items_json.ipynb` e
`notebooks/build_buildings_json.ipynb`. Este é o gate de validação que
roda depois.


In [1]:
import json
import subprocess
from pathlib import Path

REPO_ROOT = Path("../..").resolve()

items_final = json.loads((REPO_ROOT / "src/data/items.json").read_text(encoding="utf-8"))
buildings_final = json.loads((REPO_ROOT / "src/data/buildings.json").read_text(encoding="utf-8"))
combined = {**items_final, **buildings_final}

print(f"{len(items_final)} items, {len(buildings_final)} buildings, {len(combined)} combinado(s)")


1622 items, 503 buildings, 2124 combinado(s)


## Contrato com `src/lib/types.ts`

`Item` exige `name`/`base`/`ingredients`; os demais campos são opcionais
mas precisam ser um dos conhecidos - ver `src/lib/types.ts`.


In [2]:
REQUIRED_FIELDS = {"name", "base", "ingredients"}
KNOWN_FIELDS = REQUIRED_FIELDS | {"workbench", "category", "rank", "description", "price", "rarity"}

schema_errors = []
for entry_id, entry in combined.items():
    missing = REQUIRED_FIELDS - entry.keys()
    if missing:
        schema_errors.append((entry_id, "missing_fields", sorted(missing)))
    unknown = entry.keys() - KNOWN_FIELDS
    if unknown:
        schema_errors.append((entry_id, "unknown_fields", sorted(unknown)))
    if not isinstance(entry.get("name"), dict) or set(entry["name"]) != {"en", "ptBR"}:
        schema_errors.append((entry_id, "bad_name_shape", entry.get("name")))
    if not isinstance(entry.get("ingredients"), dict):
        schema_errors.append((entry_id, "bad_ingredients_shape", entry.get("ingredients")))

print(f"{len(schema_errors)} problema(s) de schema encontrado(s):")
schema_errors[:20]


0 problema(s) de schema encontrado(s):


[]

## Referências penduradas no output final


In [3]:
dangling = [
    (entry_id, ingredient_id)
    for entry_id, entry in combined.items()
    for ingredient_id in entry.get("ingredients", {})
    if ingredient_id not in combined
]
print(f"{len(dangling)} referência(s) pendurada(s):")
dangling[:20]


0 referência(s) pendurada(s):


[]

## Ciclos no grafo final


In [4]:
final_graph = {entry_id: list(entry["ingredients"].keys()) for entry_id, entry in combined.items()}

UNVISITED, IN_PROGRESS, DONE = 0, 1, 2


def find_cycles(graph):
    state = {node: UNVISITED for node in graph}
    cycles = []

    def visit(node, path):
        state[node] = IN_PROGRESS
        path.append(node)
        for dep in graph.get(node, []):
            dep_state = state.get(dep, DONE)
            if dep_state == IN_PROGRESS:
                cycle_start = path.index(dep)
                cycles.append(path[cycle_start:] + [dep])
            elif dep_state == UNVISITED:
                visit(dep, path)
        path.pop()
        state[node] = DONE

    for node in graph:
        if state[node] == UNVISITED:
            visit(node, [])
    return cycles


cycles = find_cycles(final_graph)
print(f"{len(cycles)} ciclo(s) no output final")
cycles


0 ciclo(s) no output final


[]

## Diff semântico vs HEAD

`git show` é read-only - não altera o working tree. Compara os dados por
id/campo, não texto bruto, então uma reordenação de chaves não aparece
como "mudança" falsa.


In [5]:
def git_show(path):
    result = subprocess.run(
        ["git", "show", f"HEAD:{path}"], cwd=REPO_ROOT, capture_output=True, text=True
    )
    return json.loads(result.stdout) if result.returncode == 0 else {}

prev_combined = {**git_show("src/data/items.json"), **git_show("src/data/buildings.json")}

added = set(combined) - set(prev_combined)
removed = set(prev_combined) - set(combined)
changed = {k for k in set(combined) & set(prev_combined) if combined[k] != prev_combined[k]}

print(f"+{len(added)} novo(s), -{len(removed)} removido(s), ~{len(changed)} alterado(s) vs HEAD")
print("novos:", sorted(added)[:10])
print("removidos:", sorted(removed)[:10])
print("alterados:", sorted(changed)[:10])


+0 novo(s), -0 removido(s), ~2124 alterado(s) vs HEAD
novos: []
removidos: []
alterados: ['AIcore', 'Accessory_AT_1', 'Accessory_AirDash1', 'Accessory_AirDash2', 'Accessory_AirDash3', 'Accessory_AquaResist_1', 'Accessory_Avoid_1', 'Accessory_ColdIce_1', 'Accessory_CoolResist_1', 'Accessory_DFHP_1']
